In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "1"

In [2]:
import torch
# if your GPU has less than 10GB memory, please remove the code below
# if your GPU has less than 4GB memory, use colab or kaggle instead
max_memory_gib = torch.cuda.get_device_properties('cuda').total_memory / 2 ** 30
torch.cuda.set_per_process_memory_fraction(min(1.0, 10 / max_memory_gib))
print(f"Setting memory limit to {min(1.0, 11 / max_memory_gib) * 100:.2f}%")

Setting memory limit to 70.38%


In [3]:
import transformers
import torch
model_name = "facebook/opt-1.3b"   # full model: 'facebook/opt-6.7b' or see above
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, low_cpu_mem_usage=True, torch_dtype=torch.float16, attn_implementation="flash_attention_2")

model.enable_input_require_grads()  # for gradient checkpointing compatibility, see FAQ

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


In [2]:
import random
import torch

from datasets import load_dataset
data = load_dataset("wikitext", "wikitext-2-v1")['train']
seqs = [sample['text'] for sample in data]

def get_packed_batch(size, seq_len):
    tokenizer.pad_token = tokenizer.eos_token

    batch_input_ids = []
    batch_attention_mask = []
    batch_position_ids = []

    for _ in range(size):
        packed_ids = []
        position_ids = []
        segment_mask = []

        while True:
            text = random.choice(seqs)

            tokens = tokenizer(
                text,
                add_special_tokens=False,
                return_tensors="pt"
            )["input_ids"][0]

            if len(tokens) == 0:
                continue

            seg_len = len(tokens) + 1

            if len(packed_ids) + seg_len > seq_len:
                break
            packed_ids.extend(tokens.tolist())
            packed_ids.append(tokenizer.eos_token_id)

            position_ids.extend(range(len(tokens)))
            position_ids.append(len(tokens))

            segment_mask.extend([1] * seg_len)

        pad_len = seq_len - len(packed_ids)

        packed_ids += [tokenizer.pad_token_id] * pad_len
        position_ids += [0] * pad_len
        segment_mask += [0] * pad_len

        batch_input_ids.append(torch.tensor(packed_ids))
        batch_position_ids.append(torch.tensor(position_ids))
        batch_attention_mask.append(torch.tensor(segment_mask))

    return {
        "input_ids": torch.stack(batch_input_ids),
        "attention_mask": torch.stack(batch_attention_mask),
        "position_ids": torch.stack(batch_position_ids),
    }


In [ ]:
import torch
# from torch.utils.checkpoint import checkpoint

class OffloadFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, layer, hidden_states, attention_mask, position_ids):
        ctx.layer = layer
        ctx.attention_mask = attention_mask
        ctx.position_ids = position_ids
        ctx.gpu_rng_state = torch.cuda.get_rng_state()
        layer.to("cuda")
        with torch.no_grad():
            outputs = layer(hidden_states, attention_mask=attention_mask, position_ids=position_ids)

        ctx.save_for_backward(hidden_states)
        layer.to("cpu", non_blocking=True)
        return outputs[0]

    @staticmethod
    def backward(ctx, grad_output):
        hidden_states, = ctx.saved_tensors
        layer = ctx.layer
        
        torch.cuda.set_rng_state(ctx.gpu_rng_state)
        layer.to("cuda")
        with torch.enable_grad():

            hidden_states_inc = hidden_states.detach().requires_grad_(True)
            outputs = layer(hidden_states_inc, 
                           attention_mask=ctx.attention_mask, 
                           position_ids=ctx.position_ids)[0]
            
            # Manually trigger backward for this layer
            torch.autograd.backward(outputs, grad_output)
            
        input_grad = hidden_states_inc.grad
        layer.to("cpu", non_blocking=True)
        return None, input_grad, None, None

1. Check that forward pass with offloading is `torch.allclose` to forward pass without offloading (test on 1.3B model)

In [6]:
import torch
import torch.nn.functional as F
import transformers.models.opt.modeling_opt as opt_module
from transformers.modeling_flash_attention_utils import _flash_attention_forward

setattr(opt_module, "_flash_attention_forward", _flash_attention_forward)

decoder = model.model.decoder
layers = decoder.layers
embed_tokens = decoder.embed_tokens
embed_positions = decoder.embed_positions
final_layer_norm = decoder.final_layer_norm
lm_head = model.lm_head


In [12]:
from tqdm import tqdm

def forward_pass(batch):
    input_ids = batch["input_ids"].to("cuda")
    attention_mask = batch["attention_mask"].to("cuda")
    position_ids = batch["position_ids"].to("cuda")

    embed_tokens.to("cuda")
    embed_positions.to("cuda")
    
    inputs_embeds = embed_tokens(input_ids)
    pos_embeds = embed_positions(attention_mask=attention_mask, position_ids=position_ids)
    hidden_states = inputs_embeds + pos_embeds
    
    embed_tokens.to("cpu")
    embed_positions.to("cpu")

    for layer in tqdm(layers):
        hidden_states = OffloadFunction.apply(
            layer, 
            hidden_states, 
            attention_mask, 
            position_ids
        )

    final_layer_norm.to("cuda")
    lm_head.to("cuda")
    
    hidden_states = final_layer_norm(hidden_states)
    logits = lm_head(hidden_states)
    return input_ids, logits



In [8]:
test_batch = get_packed_batch(4, 100)

In [13]:
offload_forward_output = forward_pass(test_batch)[1]

100%|██████████| 24/24 [00:02<00:00, 11.82it/s]


In [10]:
test_batch['input_ids'] = test_batch['input_ids'].to('cuda')
test_batch['attention_mask'] = test_batch['attention_mask'].to('cuda')
test_batch['position_ids'] = test_batch['position_ids'].to('cuda')
ground_truth_forward_output = model.to('cuda').forward(**test_batch).logits

In [14]:
assert torch.allclose(offload_forward_output, ground_truth_forward_output)

2. Perform forward pass with offloading

In [3]:
import transformers
import torch
model_name = "facebook/opt-6.7b"   # full model: 'facebook/opt-6.7b' or see above
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, low_cpu_mem_usage=False, torch_dtype=torch.float16, attn_implementation="flash_attention_2")

model.enable_input_require_grads()  # for gradient checkpointing compatibility, see FAQ

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
decoder = model.model.decoder
layers = decoder.layers
embed_tokens = decoder.embed_tokens
embed_positions = decoder.embed_positions
final_layer_norm = decoder.final_layer_norm
lm_head = model.lm_head

In [22]:
test_batch = get_packed_batch(4, 100)

In [23]:
logits = forward_pass(test_batch)

100%|██████████| 32/32 [00:15<00:00,  2.06it/s]


In [24]:
torch.cuda.max_memory_allocated() / 1e9

3.767724032

3. Inference the model

In [25]:
@torch.no_grad()
def generate_from_text(text, max_new_tokens=20):
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=True)
    current_ids = inputs["input_ids"]

    for _ in range(max_new_tokens):
        seq_len = current_ids.size(1)

        current_batch = {
            "input_ids": current_ids,
            "attention_mask": torch.ones((1, seq_len), dtype=torch.long),
            "position_ids": torch.arange(seq_len).unsqueeze(0)
        }

        _, logits = forward_pass(current_batch)

        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)

        current_ids = torch.cat([current_ids, next_token.cpu()], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(current_ids[0], skip_special_tokens=True)

output_text = generate_from_text("The capital of France is", max_new_tokens=10)
print(output_text)

100%|██████████| 32/32 [00:16<00:00,  1.99it/s]


The capital of France is Paris.
I'm not sure if you're


In [26]:
torch.cuda.max_memory_allocated() / 1e9

4.191801344

4. forward pass on 128x1024 

In [4]:
import torch

class OffloadFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, layer, hidden_states, attention_mask=None, position_ids=None):
        ctx.layer = layer
        ctx.attention_mask = attention_mask
        ctx.position_ids = position_ids
        ctx.gpu_rng_state = torch.cuda.get_rng_state()
        
        with torch.no_grad():
            if attention_mask is not None:
                outputs = layer(hidden_states, attention_mask=attention_mask, position_ids=position_ids)
            else:
                outputs = layer(hidden_states)

        ctx.save_for_backward(hidden_states.to('cpu', non_blocking=True))
        
        # Store whether the output was a tuple for backward
        ctx.is_tuple_output = isinstance(outputs, tuple)
        
        if ctx.is_tuple_output:
            return outputs[0]
        else:
            return outputs

    @staticmethod
    def backward(ctx, grad_output):
        hidden_states, = ctx.saved_tensors
        layer = ctx.layer
        
        torch.cuda.set_rng_state(ctx.gpu_rng_state)

        hidden_states = hidden_states.to('cuda')
        layer.to("cuda")
        
        with torch.enable_grad():
            hidden_states_inc = hidden_states.detach().requires_grad_(True)
            
            # Forward pass with gradients
            if ctx.attention_mask is not None:
                outputs = layer(hidden_states_inc, 
                              attention_mask=ctx.attention_mask, 
                              position_ids=ctx.position_ids)
            else:
                outputs = layer(hidden_states_inc)
            
            # Handle both tuple and tensor outputs correctly
            if ctx.is_tuple_output:
                output_tensor = outputs[0]
            else:
                output_tensor = outputs
            
            torch.autograd.backward(output_tensor, grad_output)
            
        input_grad = hidden_states_inc.grad
        
        # Clean up
        del hidden_states, hidden_states_inc, output_tensor
        if 'outputs' in locals():
            del outputs
            
        layer.to("cpu", non_blocking=True)
        
        return None, input_grad, None, None

In [5]:
from tqdm import tqdm

def forward_microbatch(full_batch, micro_batch_size=4):
    device = "cuda"
    num_samples = full_batch["input_ids"].size(0)
    num_mbs = num_samples // micro_batch_size

    mb_ids = torch.chunk(full_batch["input_ids"], num_mbs)
    mb_masks = torch.chunk(full_batch["attention_mask"], num_mbs)
    mb_pos = torch.chunk(full_batch["position_ids"], num_mbs)

    mb_hidden_states = []
    embed_tokens.to(device)
    embed_positions.to(device)
    
    for i in range(num_mbs):
        h = embed_tokens(mb_ids[i].to(device)) + \
            embed_positions(attention_mask=mb_masks[i].to(device), position_ids=mb_pos[i].to(device))
        mb_hidden_states.append(h)
        
    embed_tokens.to("cpu", non_blocking=True)
    embed_positions.to("cpu", non_blocking=True)

    for i, layer in enumerate(tqdm(layers, desc="Forward Layers")):
        layer.to(device)
        if i != len(layers) - 1: layers[i + 1].to(device, non_blocking=True)
        new_states = []
        for i in range(num_mbs):
            h_out = OffloadFunction.apply(layer, mb_hidden_states[i], mb_masks[i].to(device), mb_pos[i].to(device))
            new_states.append(h_out)
        
        mb_hidden_states = new_states
        layer.to("cpu", non_blocking=True)
        if i != len(layers) - 1: layers[i + 1].to(device, non_blocking=True)

    final_layer_norm.to(device)
    lm_head.to(device)
    
    all_logits = []
    for i in range(num_mbs):
        h = OffloadFunction.apply(final_layer_norm, mb_hidden_states[i])
        logits = OffloadFunction.apply(lm_head, h)
        all_logits.append(logits.to('cpu', non_blocking=True))

    return mb_ids, all_logits

In [6]:
large_batch = get_packed_batch(128, 1024)

In [16]:
_, logits = forward_microbatch(large_batch, micro_batch_size=8)

Forward Layers: 100%|██████████| 32/32 [00:57<00:00,  1.81s/it]


In [17]:
torch.cuda.max_memory_allocated() / 1e9

4.844896256

5. Train step with Lora

In [8]:
torch.cuda.memory._record_memory_history()

In [10]:
from peft import LoraConfig, get_peft_model


config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)

model.print_trainable_parameters()

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to("cuda")

trainable params: 4,194,304 || all params: 6,662,668,288 || trainable%: 0.0630


/home/korenikil/efficient-dl-systems/.venv/lib/python3.10/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/korenikil/efficient-dl-systems/.venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [9]:
decoder = model.base_model.model.model.decoder
layers = decoder.layers
embed_tokens = decoder.embed_tokens
embed_positions = decoder.embed_positions
final_layer_norm = decoder.final_layer_norm
lm_head = model.base_model.model.lm_head

In [11]:
def train_step_microbatch(full_batch, optimizer, mb_size=4):
    mb_ids, mb_logits = forward_microbatch(full_batch, micro_batch_size=mb_size)
    num_mbs = len(mb_ids)
    
    total_loss = 0
    
    for i in range(num_mbs):
        logits = mb_logits[i]
        labels = mb_ids[i].to("cuda")
        
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        
        flat_logits = shift_logits.view(-1, shift_logits.size(-1))
        flat_labels = shift_labels.view(-1)

        loss = torch.nn.functional.cross_entropy(
            flat_logits.to('cuda'), 
            flat_labels
        ) / num_mbs

        loss.backward(retain_graph=True if i < num_mbs-1 else False)
        total_loss += loss.cpu().item()
        
    
    optimizer.step()
    optimizer.zero_grad()
    return total_loss

In [12]:
model.train()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

large_batch = get_packed_batch(128, 1024)
for _ in range(10):
    loss_train = train_step_microbatch(large_batch, optimizer, 8)
    print(f"Loss: {loss_train}")

Forward Layers: 100%|██████████| 32/32 [00:54<00:00,  1.71s/it]


Loss: 3.745361328125


Forward Layers: 100%|██████████| 32/32 [00:45<00:00,  1.43s/it]


Loss: 3.7403564453125


Forward Layers: 100%|██████████| 32/32 [00:45<00:00,  1.43s/it]


...

Loss: 3.7152099609375


Forward Layers: 100%|██████████| 32/32 [00:45<00:00,  1.43s/it]


Loss: 3.7218017578125


Forward Layers: 100%|██████████| 32/32 [00:45<00:00,  1.42s/it]


Loss: 3.7073974609375


In [13]:
torch.cuda.memory._dump_snapshot(f"my_snapshot.pickle")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [14]:
!python -m torch.cuda._memory_viz trace_plot my_snapshot.pickle -o snapshot_visualization.html

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/usr/lib/python3.10/runpy.py:126: RuntimeWarning: 'torch.cuda._memory_viz' found in sys.modules after import of package 'torch.cuda', but prior to execution of 'torch.cuda._memory_viz'; this may result in unpredictable behaviour
  warn(RuntimeWarning(msg))


In [15]:
torch.cuda.max_memory_allocated() / 1e9

5.251325952